# AetherCell — delta_z Embedding Generator

This notebook generates **delta_z** embeddings: 256-dimensional latent-space perturbation
vectors that capture the effect of a **drug** or **shRNA** knockdown on a specific cell line.

## What is delta_z?

```
z_ctrl  = Encoder(control_expression)          # control state in latent space
delta_z = DeltaHead([drug/shRNA × cell, z_ctrl])  # perturbation direction
z_pred  = z_ctrl + delta_z                     # predicted post-perturbation state
```

`delta_z` is the **directional shift** caused by a perturbation and can serve as a
feature embedding for downstream tasks such as:
- Drug mechanism clustering / similarity search
- Drug sensitivity (IC50) prediction
- Multi-task learning inputs
- Cross-perturbation comparison

## Supported Modes

| Perturbation | Control source | L1000 required | RNAseq required |
|---|---|---|---|
| Drug (compound) | L1000 | Yes | Yes |
| Drug (compound) | RNAseq | **No** | Yes |
| shRNA knockdown | L1000 | Yes | Yes |
| shRNA knockdown | RNAseq | **No** | Yes |

> **RNAseq control mode:** The RNA encoder produces z_ctrl from RNAseq instead of
> the LINCS encoder reading L1000. This is convenient when L1000 data is unavailable.
> Note that it introduces a distributional shift relative to the training setup,
> which used L1000 for control encoding.


## 1. Environment Setup

In [ ]:
import os
import sys
import pickle
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# Optional visualisation dependencies
try:
    import matplotlib.pyplot as plt
    from sklearn.decomposition import PCA
    VIZ_AVAILABLE = True
except ImportError:
    VIZ_AVAILABLE = False
    print("[INFO] matplotlib / scikit-learn not found — visualisation cells will be skipped.")

# ---------------------------------------------------------------------------
# Add the src/ directory (containing aethercell_drug.py etc.) to Python path.
# By default we assume the notebook lives inside src/.
# If you placed it elsewhere, set SRC_DIR to the absolute path of src/.
# ---------------------------------------------------------------------------
SRC_DIR = os.getcwd()
# SRC_DIR = "/absolute/path/to/aethercell_git/src"   # <-- uncomment & edit if needed

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# AetherCell model modules
from LINCSvae import LINCSVAE
from RNAvae import RNAVAE
from aethercell_drug import JointPerturbationPredictor
from aethercell_sh import JointPerturbationPredictor_sh
from dataloader_all import PredictorDatasetDP2_i, PredictorDatasetDP2_sh_i

print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

## 2. Configuration

**Edit only this cell** before running the notebook.

### Required files — always
| Variable | Description |
|---|---|
| `LINCSVAE_CKPT` | Checkpoint of the pre-trained LINCS VAE |
| `RNAVAE_CKPT` | Checkpoint of the pre-trained RNA VAE |
| `META_CSV` | Metadata CSV (see column requirements below) |
| `RNASEQ_PARQUET` | RNAseq expression matrix — shape `(genes, cell_lines)` |

### Required files — drug mode
| Variable | Description |
|---|---|
| `DRUG_MODEL_CKPT` | Trained drug perturbation model checkpoint |
| `MOLFORMER_PATH` | Directory of MolFormer tokeniser & weights |
| `DRUG_INPUT_IDS_NPY` | Pre-tokenised SMILES input IDs `(N_drugs, 160)` |
| `DRUG_ATTN_MASK_NPY` | Corresponding attention masks |
| `DRUG_IDX_MAP_PKL` | `dict` mapping `pert_id -> row index` |

### Required files — shRNA mode
| Variable | Description |
|---|---|
| `SHRNA_MODEL_CKPT` | Trained shRNA perturbation model checkpoint |
| `SH_PPI_CSV` | PPI-based gene embeddings CSV (index = `gene_ensg`) |
| `SH_SEQ_NPY` | Sequence embeddings `(N_genes, 1152)` |
| `SH_SEQ_PKL` | `dict` mapping `gene_ensg -> row index` |

### Required files — L1000 control source
| Variable | Description |
|---|---|
| `L1000_CTRL_NPY` | L1000 control expression matrix `(N_ctrl, 978)` |
| `CTRL_IDX_MAP_PKL` | `dict` mapping `control_id -> row index` |

### Meta CSV column requirements
| Mode | Required columns |
|---|---|
| Drug + L1000 | `sample_id`, `pert_id`, `cell_iname`, `representative_control` |
| Drug + RNAseq | `sample_id`, `pert_id`, `cell_iname` |
| shRNA + L1000 | `sample_id`, `gene_ensg`, `cell_iname`, `representative_control` |
| shRNA + RNAseq | `sample_id`, `gene_ensg`, `cell_iname` |

In [ ]:
# ===========================================================================
#                          USER CONFIGURATION
# ===========================================================================

# -- Mode selection ----------------------------------------------------------
PERTURBATION_MODE = "drug"    # "drug"  |  "shrna"
CONTROL_SOURCE    = "L1000"   # "L1000" |  "RNAseq"

# -- Hardware ----------------------------------------------------------------
DEVICE      = "cuda:0" if torch.cuda.is_available() else "cpu"
BATCH_SIZE  = 256             # reduce to 64 / 32 if GPU memory is limited
NUM_WORKERS = 4               # set to 0 on Windows if DataLoader hangs

# -- Output ------------------------------------------------------------------
OUTPUT_DIR = "./delta_z_output"

# -- Shared VAE checkpoints --------------------------------------------------
LINCSVAE_CKPT = "../result/model_ckpt_L_random/epoch_184.pt"
RNAVAE_CKPT   = "../result/model_checkpoints_RNAseq/best_model.pt"

# -- Shared data paths -------------------------------------------------------
META_CSV       = "/path/to/meta.csv"
RNASEQ_PARQUET = "/path/to/RNAseq.parquet"

# -- Drug-specific paths (only needed when PERTURBATION_MODE = "drug") -------
DRUG_MODEL_CKPT    = "/path/to/drug_model/best_model.pt"
MOLFORMER_PATH     = "./mini_molformer"
DRUG_INPUT_IDS_NPY = "/path/to/drug_input_ids.npy"
DRUG_ATTN_MASK_NPY = "/path/to/drug_attention_mask.npy"
DRUG_IDX_MAP_PKL   = "/path/to/drug_idx_map.pkl"

# -- shRNA-specific paths (only needed when PERTURBATION_MODE = "shrna") -----
SHRNA_MODEL_CKPT = "/path/to/shrna_model/best_model.pt"
SH_PPI_CSV       = "/path/to/ensg_PPI_emb.csv"
SH_SEQ_NPY       = "/path/to/emb_tokens_first_all.npy"
SH_SEQ_PKL       = "/path/to/id2idx_ensg_seq2_all.pkl"

# -- L1000 control paths (only needed when CONTROL_SOURCE = "L1000") ---------
L1000_CTRL_NPY   = "/path/to/L1000_ctrl.npy"
CTRL_IDX_MAP_PKL = "/path/to/ctrl_idx_map.pkl"

# ===========================================================================
print(f"Perturbation mode : {PERTURBATION_MODE}")
print(f"Control source    : {CONTROL_SOURCE}")
print(f"Device            : {DEVICE}")
print(f"Output directory  : {OUTPUT_DIR}")

## 3. Helper Utilities

This section defines:
- A utility to strip DataParallel `module.` prefixes from saved checkpoints
- Custom `Dataset` classes for the **RNAseq-only control** modes (no L1000 needed)
- A unified `generate_delta_z()` inference function covering all four modes

In [ ]:
# ---------------------------------------------------------------------------
# Checkpoint utility
# ---------------------------------------------------------------------------

def remove_module_prefix(state_dict: dict) -> dict:
    """
    Strip the 'module.' prefix that nn.DataParallel adds to parameter names.
    Safe to call even when the prefix is absent.
    """
    return {
        (k[7:] if k.startswith("module.") else k): v
        for k, v in state_dict.items()
    }


# ---------------------------------------------------------------------------
# Dataset: Drug perturbation — RNAseq-only control (no L1000)
# ---------------------------------------------------------------------------

class DrugRNAseqDataset(Dataset):
    """
    Dataset for drug perturbation inference when CONTROL_SOURCE = "RNAseq".
    No L1000 data is needed.

    The RNAseq embedding is used for both:
      - cell identity  (query in cross-attention)
      - control state  (z_ctrl_cond input to delta head)

    Required meta CSV columns:
        sample_id   -- unique identifier for this (cell, drug) pair
        pert_id     -- drug identifier (must exist in drug_idx_map)
        cell_iname  -- cell line name (must match a column in RNAseq.parquet)
    """

    def __init__(
        self,
        meta_csv: str,
        RNA_parquet_path: str,
        drug_input_ids_npy: str,
        drug_attention_mask_npy: str,
        drug_idx_map_path: str,
    ):
        self.meta = pd.read_csv(meta_csv, engine="python")

        # RNAseq matrix -- shape (G, C): genes x cell_lines
        df_rna = pd.read_parquet(RNA_parquet_path)
        self.RNA_arr      = df_rna.values.astype(np.float32)
        self.cell_to_idx  = {c: i for i, c in enumerate(df_rna.columns)}
        del df_rna

        # Pre-tokenised drug arrays
        arr_ids  = np.load(drug_input_ids_npy,      mmap_mode="r")
        arr_mask = np.load(drug_attention_mask_npy, mmap_mode="r")
        self.drug_input_ids      = torch.from_numpy(arr_ids.astype(np.int64))
        self.drug_attention_mask = torch.from_numpy(arr_mask.astype(np.int64))

        with open(drug_idx_map_path, "rb") as f:
            self.drug_idx_map = pickle.load(f)

    def __len__(self):
        return len(self.meta)

    def __getitem__(self, idx):
        row = self.meta.iloc[idx]

        col_idx = self.cell_to_idx[row["cell_iname"]]
        rna_vec = self.RNA_arr[:, col_idx]   # (G,)

        pert_id = str(row["pert_id"])
        d_idx   = self.drug_idx_map[pert_id]

        return {
            "sample_id":      str(row.get("sample_id", idx)),
            "pert_id":        pert_id,
            "cell_id":        row["cell_iname"],
            "rna":            torch.from_numpy(rna_vec),
            "input_ids":      self.drug_input_ids[d_idx],
            "attention_mask": self.drug_attention_mask[d_idx],
        }


# ---------------------------------------------------------------------------
# Dataset: shRNA perturbation — RNAseq-only control (no L1000)
# ---------------------------------------------------------------------------

class ShRNARNAseqDataset(Dataset):
    """
    Dataset for shRNA perturbation inference when CONTROL_SOURCE = "RNAseq".
    No L1000 data is needed.

    Required meta CSV columns:
        sample_id   -- unique identifier for this (cell, gene) pair
        gene_ensg   -- Ensembl gene ID (must exist in PPI / seq embeddings)
        cell_iname  -- cell line name (must match a column in RNAseq.parquet)
    """

    def __init__(
        self,
        meta_csv: str,
        RNA_parquet_path: str,
        sh_embed_PPI_csv: str,
        sh_seq_emb_npy: str,
        sh_seq_emb_pkl: str,
    ):
        self.meta = pd.read_csv(meta_csv, engine="python")

        # RNAseq matrix
        df_rna = pd.read_parquet(RNA_parquet_path)
        self.RNA_arr     = df_rna.values.astype(np.float32)
        self.cell_to_idx = {c: i for i, c in enumerate(df_rna.columns)}
        del df_rna

        # PPI embeddings -- CSV with index = gene_ensg
        df_ppi   = pd.read_csv(sh_embed_PPI_csv, engine="python")
        emb_cols = [c for c in df_ppi.columns if c not in {"gene", "pert_id"}]
        df_ppi   = df_ppi.set_index("gene")[emb_cols].astype(np.float32)
        self.sh_PPI_arr    = df_ppi.to_numpy(copy=False)
        self.sh_PPI_id2row = {g: i for i, g in enumerate(df_ppi.index.astype(str))}
        del df_ppi

        # Sequence embeddings
        self.sh_seq        = np.load(sh_seq_emb_npy)
        self.sh_seq_id2idx = pd.read_pickle(sh_seq_emb_pkl)

    def __len__(self):
        return len(self.meta)

    def __getitem__(self, idx):
        row = self.meta.iloc[idx]

        col_idx = self.cell_to_idx[row["cell_iname"]]
        rna_vec = self.RNA_arr[:, col_idx]   # (G,)

        gene    = str(row["gene_ensg"])
        ppi_emb = self.sh_PPI_arr[self.sh_PPI_id2row[gene], :].astype(np.float32)
        seq_vec = self.sh_seq[self.sh_seq_id2idx[gene], :].astype(np.float32)

        return {
            "sample_id":        str(row.get("sample_id", idx)),
            "pert_id":          gene,
            "cell_id":          row["cell_iname"],
            "rna":              torch.from_numpy(rna_vec),
            "sh_PPI_embedding": torch.from_numpy(ppi_emb),
            "sh_seq_embedding": torch.from_numpy(seq_vec),
        }


print("Helper utilities and dataset classes defined.")

In [ ]:
# ---------------------------------------------------------------------------
# Unified inference function
# ---------------------------------------------------------------------------

@torch.no_grad()
def generate_delta_z(
    model,
    dataloader: DataLoader,
    perturbation_mode: str,
    control_source: str,
    device: str,
):
    """
    Run batched inference and collect delta_z embeddings.

    Parameters
    ----------
    model : JointPerturbationPredictor | JointPerturbationPredictor_sh
        Loaded, eval-mode perturbation model.
    dataloader : DataLoader
        Yields batches from one of the four dataset classes.
    perturbation_mode : {"drug", "shrna"}
    control_source : {"L1000", "RNAseq"}
    device : str
        Target device string, e.g. "cuda:0" or "cpu".

    Returns
    -------
    delta_z_array : np.ndarray, shape (N, 256)
    metadata : dict with lists: sample_id, pert_id, cell_id, control_id
    """
    model.eval()

    all_delta_z = []
    meta = {"sample_id": [], "pert_id": [], "cell_id": [], "control_id": []}

    for batch in tqdm(dataloader, desc="Generating delta_z"):
        rna = batch["rna"].to(device).float()
        B   = rna.shape[0]

        # ----------------------------------------------------------------
        # Drug perturbation
        # ----------------------------------------------------------------
        if perturbation_mode == "drug":
            input_ids = batch["input_ids"].to(device)
            attn_mask = batch["attention_mask"].to(device)

            if control_source == "L1000":
                # Standard path: L1000 control -> LINCSencoder -> z_ctrl_cond
                # This exactly matches the training setup.
                control = batch["control"].to(device).float()
                _, _, delta_z, _ = model(rna, control, input_ids, attn_mask)

            else:  # control_source == "RNAseq"
                # Alternative path: RNAseq -> RNAencoder -> z_ctrl_cond
                # No L1000 data is required.
                # Step 1: encode drug through MolFormer
                drug_out    = model.molformer(
                    input_ids=input_ids, attention_mask=attn_mask
                )
                drug_tokens = model.mlp_tokens(
                    drug_out.last_hidden_state
                )                                         # (B, T, 256)

                # Step 2: encode cell state via RNAencoder
                _, cell_mu, _ = model.RNAencoder(rna)     # (B, 256)

                # Step 3: reuse cell_mu as z_ctrl_cond (no L1000 needed)
                z_ctrl_cond = cell_mu

                # Step 4: cross-attention — drug tokens queried by cell identity
                attn_fused = model.cross_attention(
                    drug_tokens, cell_mu
                )                                         # (B, 256)

                # Step 5: delta head — fused drug-cell + control -> delta_z
                delta_z = model.delta_head(
                    torch.cat([attn_fused, z_ctrl_cond], dim=1)
                )                                         # (B, 256)

        # ----------------------------------------------------------------
        # shRNA perturbation
        # ----------------------------------------------------------------
        elif perturbation_mode == "shrna":
            ppi_emb = batch["sh_PPI_embedding"].to(device).float()
            seq_emb = batch["sh_seq_embedding"].to(device).float()

            if control_source == "L1000":
                # Standard path
                control = batch["control"].to(device).float()
                _, _, delta_z, _ = model(rna, control, ppi_emb, seq_emb)

            else:  # control_source == "RNAseq"
                # Step 1: encode cell state via RNAencoder
                _, cell_mu, _ = model.RNAencoder(rna)     # (B, 256)

                # Step 2: reuse cell_mu as z_ctrl_cond
                z_ctrl_cond = cell_mu

                # Step 3: tokenise shRNA embeddings
                # Note: the attribute is named 'tokenrize' in the source code
                sh_tokens = model.tokenrize(ppi_emb, seq_emb)  # (B, 1+t_seq, 256)

                # Step 4: cross-attention — shRNA tokens queried by cell identity
                attn_fused = model.cross_attention(
                    sh_tokens, cell_mu
                )                                              # (B, 256)

                # Step 5: delta head
                delta_z = model.delta_head(
                    torch.cat([attn_fused, z_ctrl_cond], dim=1)
                )                                              # (B, 256)

        else:
            raise ValueError(
                f"Unknown perturbation_mode: {perturbation_mode!r}. "
                "Choose 'drug' or 'shrna'."
            )

        all_delta_z.append(delta_z.cpu().numpy())

        # Accumulate metadata
        meta["sample_id"].extend(batch.get("sample_id",  [""] * B))
        meta["pert_id"].extend(batch.get("pert_id",      [""] * B))
        meta["cell_id"].extend(batch.get("cell_id",      [""] * B))
        meta["control_id"].extend(batch.get("control_id", ["N/A"] * B))

    delta_z_array = np.concatenate(all_delta_z, axis=0)  # (N, 256)
    return delta_z_array, meta


print("Inference function defined.")

## 4. Load Models

In [ ]:
# ---------------------------------------------------------------------------
# Load LINCS VAE (encoder + decoder both needed by the perturbation model)
# ---------------------------------------------------------------------------
print("Loading LINCS VAE ...")
LINvae_model = LINCSVAE("cpu")
lin_ckpt = torch.load(LINCSVAE_CKPT, map_location="cpu", weights_only=False)
LINvae_model.load_state_dict(
    remove_module_prefix(lin_ckpt["vae_model_state_dict"])
)
lincs_encoder = LINvae_model.encoder.to(DEVICE)
lincs_decoder = LINvae_model.decoder.to(DEVICE)
lincs_encoder.eval()
lincs_decoder.eval()
print(f"  Loaded from: {LINCSVAE_CKPT}")

# ---------------------------------------------------------------------------
# Load RNA VAE (encoder only; decoder is not used at inference time)
# ---------------------------------------------------------------------------
print("Loading RNA VAE ...")
RNAvae_model = RNAVAE("cpu")
rna_ckpt = torch.load(RNAVAE_CKPT, map_location="cpu", weights_only=False)
RNAvae_model.load_state_dict(
    remove_module_prefix(rna_ckpt["vae_model_state_dict"])
)
rna_encoder = RNAvae_model.encoder.to(DEVICE)
rna_encoder.eval()
print(f"  Loaded from: {RNAVAE_CKPT}")

In [ ]:
# ---------------------------------------------------------------------------
# Build the perturbation model and load its trained weights
# ---------------------------------------------------------------------------
print(f"Building {PERTURBATION_MODE.upper()} perturbation model ...")

if PERTURBATION_MODE == "drug":
    model = JointPerturbationPredictor(
        LINCSencoder  = lincs_encoder,
        LINCSdecoder  = lincs_decoder,
        RNAencoder    = rna_encoder,
        molformer_path = MOLFORMER_PATH,
        device         = DEVICE,
    ).to(DEVICE)

    print(f"  Loading weights: {DRUG_MODEL_CKPT}")
    ckpt = torch.load(DRUG_MODEL_CKPT, map_location=DEVICE, weights_only=False)
    model.load_state_dict(remove_module_prefix(ckpt["model_state_dict"]))

elif PERTURBATION_MODE == "shrna":
    model = JointPerturbationPredictor_sh(
        LINCSencoder = lincs_encoder,
        LINCSdecoder = lincs_decoder,
        RNAencoder   = rna_encoder,
        device       = DEVICE,
    ).to(DEVICE)

    print(f"  Loading weights: {SHRNA_MODEL_CKPT}")
    ckpt = torch.load(SHRNA_MODEL_CKPT, map_location=DEVICE, weights_only=False)
    model.load_state_dict(remove_module_prefix(ckpt["model_state_dict"]))

else:
    raise ValueError(
        f"Unknown PERTURBATION_MODE: {PERTURBATION_MODE!r}. "
        "Choose 'drug' or 'shrna'."
    )

model.eval()

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total parameters     : {total:,}")
print(f"  Trainable parameters : {trainable:,}")
print("Model ready.")

## 5. Prepare Dataset & DataLoader

The correct dataset class is selected automatically based on `PERTURBATION_MODE`
and `CONTROL_SOURCE`.

In [ ]:
print(f"Preparing dataset  [{PERTURBATION_MODE} | control={CONTROL_SOURCE}] ...")

# ---- Drug + L1000 control --------------------------------------------------
if PERTURBATION_MODE == "drug" and CONTROL_SOURCE == "L1000":
    # Uses the existing PredictorDatasetDP2_i class from dataloader_all.py.
    # Required meta CSV columns:
    #   sample_id, pert_id, cell_iname, representative_control
    dataset = PredictorDatasetDP2_i(
        meta_csv                = META_CSV,
        L1000_ctrl_npy          = L1000_CTRL_NPY,
        ctrl_idx_map_path       = CTRL_IDX_MAP_PKL,
        RNA_parquet_path        = RNASEQ_PARQUET,
        drug_input_ids_npy      = DRUG_INPUT_IDS_NPY,
        drug_attention_mask_npy = DRUG_ATTN_MASK_NPY,
        drug_idx_map_path       = DRUG_IDX_MAP_PKL,
    )

# ---- Drug + RNAseq control (no L1000 needed) --------------------------------
elif PERTURBATION_MODE == "drug" and CONTROL_SOURCE == "RNAseq":
    # Uses the custom DrugRNAseqDataset defined above.
    # Required meta CSV columns: sample_id, pert_id, cell_iname
    dataset = DrugRNAseqDataset(
        meta_csv                = META_CSV,
        RNA_parquet_path        = RNASEQ_PARQUET,
        drug_input_ids_npy      = DRUG_INPUT_IDS_NPY,
        drug_attention_mask_npy = DRUG_ATTN_MASK_NPY,
        drug_idx_map_path       = DRUG_IDX_MAP_PKL,
    )

# ---- shRNA + L1000 control --------------------------------------------------
elif PERTURBATION_MODE == "shrna" and CONTROL_SOURCE == "L1000":
    # Uses the existing PredictorDatasetDP2_sh_i class from dataloader_all.py.
    # Required meta CSV columns:
    #   sample_id, gene_ensg, cell_iname, representative_control
    dataset = PredictorDatasetDP2_sh_i(
        meta_csv          = META_CSV,
        L1000_ctrl_npy    = L1000_CTRL_NPY,
        ctrl_idx_map_path = CTRL_IDX_MAP_PKL,
        RNA_parquet_path  = RNASEQ_PARQUET,
        sh_embed_PPI_csv  = SH_PPI_CSV,
        sh_seq_emb_npy    = SH_SEQ_NPY,
        sh_seq_emb_pkl    = SH_SEQ_PKL,
    )

# ---- shRNA + RNAseq control (no L1000 needed) -------------------------------
elif PERTURBATION_MODE == "shrna" and CONTROL_SOURCE == "RNAseq":
    # Uses the custom ShRNARNAseqDataset defined above.
    # Required meta CSV columns: sample_id, gene_ensg, cell_iname
    dataset = ShRNARNAseqDataset(
        meta_csv         = META_CSV,
        RNA_parquet_path = RNASEQ_PARQUET,
        sh_embed_PPI_csv = SH_PPI_CSV,
        sh_seq_emb_npy   = SH_SEQ_NPY,
        sh_seq_emb_pkl   = SH_SEQ_PKL,
    )

else:
    raise ValueError(
        f"Unsupported combination: PERTURBATION_MODE={PERTURBATION_MODE!r}, "
        f"CONTROL_SOURCE={CONTROL_SOURCE!r}"
    )

print(f"  Dataset size: {len(dataset):,} samples")

# Preview first sample
print("\nFirst sample preview:")
sample = dataset[0]
for k, v in sample.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:20s}: Tensor{tuple(v.shape)}, dtype={v.dtype}")
    else:
        print(f"  {k:20s}: {v!r}")

In [ ]:
dataloader = DataLoader(
    dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = False,          # preserve sample order for metadata alignment
    num_workers = NUM_WORKERS,
    pin_memory  = torch.cuda.is_available(),
)

print(f"DataLoader ready — {len(dataloader)} batches of up to {BATCH_SIZE} samples.")

## 6. Run Inference — Generate delta_z

In [ ]:
delta_z_array, metadata = generate_delta_z(
    model             = model,
    dataloader        = dataloader,
    perturbation_mode = PERTURBATION_MODE,
    control_source    = CONTROL_SOURCE,
    device            = DEVICE,
)

print(f"\ndelta_z shape : {delta_z_array.shape}   (N_samples x latent_dim)")
print(f"dtype         : {delta_z_array.dtype}")
print(f"value range   : [{delta_z_array.min():.4f}, {delta_z_array.max():.4f}]")

## 7. Save Results

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Raw numpy array
npy_path = os.path.join(OUTPUT_DIR, "delta_z.npy")
np.save(npy_path, delta_z_array)
print(f"Saved raw array  : {npy_path}  {delta_z_array.shape}")

# 2. CSV with metadata + flattened delta_z columns (dz_0 ... dz_255)
n_dim = delta_z_array.shape[1]
result_df = pd.DataFrame({
    "sample_id":  metadata["sample_id"],
    "pert_id":    metadata["pert_id"],
    "cell_id":    metadata["cell_id"],
    "control_id": metadata["control_id"],
})
for i in range(n_dim):
    result_df[f"dz_{i}"] = delta_z_array[:, i]

csv_path = os.path.join(OUTPUT_DIR, "delta_z.csv")
result_df.to_csv(csv_path, index=False)
print(f"Saved CSV        : {csv_path}  {result_df.shape}")

# 3. Metadata-only CSV (convenient for external joins)
meta_path = os.path.join(OUTPUT_DIR, "metadata.csv")
result_df[["sample_id", "pert_id", "cell_id", "control_id"]].to_csv(
    meta_path, index=False
)
print(f"Saved metadata   : {meta_path}")

print(f"\nAll files saved to: {OUTPUT_DIR}")

## 8. Inspect Results (Optional)

The cells below print summary statistics and generate a PCA visualisation.
They can be skipped if you only need the saved files.

In [ ]:
# ---- Basic statistics ------------------------------------------------------
N, D   = delta_z_array.shape
norms  = np.linalg.norm(delta_z_array, axis=1)   # L2 norm per sample
dim_var = delta_z_array.var(axis=0)               # variance per dimension

print("=" * 45)
print(" delta_z Summary Statistics")
print("=" * 45)
print(f"  Samples (N)     : {N:,}")
print(f"  Latent dim (D)  : {D}")
print(f"  Value mean      : {delta_z_array.mean():.6f}")
print(f"  Value std       : {delta_z_array.std():.6f}")
print(f"  L2 norm — mean  : {norms.mean():.4f} ± {norms.std():.4f}")
print(f"  L2 norm — min   : {norms.min():.4f}")
print(f"  L2 norm — max   : {norms.max():.4f}")

top5 = np.argsort(dim_var)[::-1][:5]
print(f"\n  Top-5 highest-variance dimensions:")
for rank, d in enumerate(top5, 1):
    print(f"    #{rank}: dim {d:3d}  var={dim_var[d]:.4f}")
print("=" * 45)

In [ ]:
# ---- PCA visualisation (requires matplotlib + scikit-learn) ----------------
if not VIZ_AVAILABLE:
    print("Skipped: install matplotlib and scikit-learn to enable visualisation.\n"
          "  pip install matplotlib scikit-learn")
else:
    pca    = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(delta_z_array)   # (N, 2)
    ev     = pca.explained_variance_ratio_

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # -- Scatter: PC1 vs PC2, coloured by cell line --------------------------
    unique_cells = sorted(set(metadata["cell_id"]))
    cmap         = plt.get_cmap("tab20", max(len(unique_cells), 1))
    cell2color   = {c: cmap(i) for i, c in enumerate(unique_cells)}
    colors       = [cell2color[c] for c in metadata["cell_id"]]

    ax = axes[0]
    ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=10, alpha=0.7)
    ax.set_xlabel(f"PC 1 ({ev[0]*100:.1f}%)")
    ax.set_ylabel(f"PC 2 ({ev[1]*100:.1f}%)")
    ax.set_title("PCA of delta_z — coloured by cell line")

    from matplotlib.patches import Patch
    shown = unique_cells[:15]
    handles = [Patch(color=cell2color[c], label=c) for c in shown]
    if len(unique_cells) > 15:
        handles.append(
            Patch(color="white", label=f"... +{len(unique_cells)-15} more")
        )
    ax.legend(handles=handles, fontsize=7, loc="best", framealpha=0.6)

    # -- L2 norm distribution ------------------------------------------------
    ax2 = axes[1]
    ax2.hist(norms, bins=50, color="steelblue", edgecolor="white", linewidth=0.5)
    ax2.axvline(
        norms.mean(), color="tomato", linestyle="--",
        linewidth=1.5, label=f"mean = {norms.mean():.2f}"
    )
    ax2.set_xlabel("L2 norm of delta_z")
    ax2.set_ylabel("Count")
    ax2.set_title("Distribution of delta_z L2 norms")
    ax2.legend()

    plt.tight_layout()
    fig_path = os.path.join(OUTPUT_DIR, "delta_z_pca.png")
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Figure saved: {fig_path}")

## Appendix A — Using the Saved Embeddings

```python
import numpy as np
import pandas as pd

# Option 1: load raw array
delta_z = np.load("./delta_z_output/delta_z.npy")  # (N, 256)

# Option 2: load with metadata
df      = pd.read_csv("./delta_z_output/delta_z.csv")
dz_cols = [c for c in df.columns if c.startswith("dz_")]
delta_z = df[dz_cols].values                         # (N, 256)
meta    = df[["sample_id", "pert_id", "cell_id"]]
```

### Output file summary

| File | Shape / Description |
|---|---|
| `delta_z.npy` | Raw numpy array `(N, 256)` |
| `delta_z.csv` | Metadata + 256 `dz_*` columns |
| `metadata.csv` | `sample_id`, `pert_id`, `cell_id`, `control_id` only |
| `delta_z_pca.png` | PCA scatter + L2 norm histogram (if matplotlib available) |

## Appendix B — Tokenise New Drug SMILES

If your drugs are **not** in the pre-tokenised arrays, run this cell to tokenise
new SMILES strings on the fly and create the required `.npy` / `.pkl` files.

In [ ]:
# ---- Tokenise a list of SMILES strings and export to .npy + .pkl -----------
# Uncomment and adapt to your needs.

# from transformers import AutoTokenizer
#
# SMILES_LIST = [
#     "CC(=O)Oc1ccccc1C(=O)O",   # aspirin
#     "CN1CCC[C@H]1c2cccnc2",    # nicotine
#     # add more SMILES here ...
# ]
# PERT_IDS = ["aspirin", "nicotine"]  # must match pert_id in your meta CSV
#
# tokenizer = AutoTokenizer.from_pretrained(MOLFORMER_PATH, trust_remote_code=True)
#
# MAX_LENGTH = 160
# input_ids_list, attn_mask_list = [], []
#
# for smiles in SMILES_LIST:
#     enc = tokenizer(
#         smiles,
#         return_tensors="pt",
#         padding="max_length",
#         truncation=True,
#         max_length=MAX_LENGTH,
#     )
#     input_ids_list.append(enc["input_ids"].squeeze(0).numpy())
#     attn_mask_list.append(enc["attention_mask"].squeeze(0).numpy())
#
# input_ids_arr = np.stack(input_ids_list).astype(np.int64)    # (N, 160)
# attn_mask_arr = np.stack(attn_mask_list).astype(np.int64)    # (N, 160)
# idx_map       = {pid: i for i, pid in enumerate(PERT_IDS)}
#
# OUT = "./my_drug_tokens"
# os.makedirs(OUT, exist_ok=True)
# np.save(os.path.join(OUT, "drug_input_ids.npy"), input_ids_arr)
# np.save(os.path.join(OUT, "drug_attention_mask.npy"), attn_mask_arr)
# with open(os.path.join(OUT, "drug_idx_map.pkl"), "wb") as f:
#     pickle.dump(idx_map, f)
# print(f"Tokenised {len(SMILES_LIST)} drugs -> {OUT}")

print("Appendix B cell is ready. Uncomment the code above to tokenise new SMILES.")